# Session 1 - pairing gate + parsing


**CPU, 1-3 h.** Verifies that every manipulated clip has a frame-aligned authentic
counterpart, then parses the paired clips into 384 px face crops with region label maps.

Inputs: the code dataset (`<your-code-dataset>`) and a FaceForensics++ mirror
(`<your-ff-mirror>`) or a Celeb-DF mirror (`<your-celebdf-mirror>`).
Accelerator **None**, Internet **On** (mediapipe and the landmarker model are downloaded).

The output of this session is the dataset that every later session reads.

In [ ]:
SESSION = "S1 gate+parse"

# ============================== CONFIG ==============================
DATA            = ""     # "" = auto-detect the folder containing original/
LAYOUT          = "auto" # auto | ffpp | celebdf
METHODS         = ""     # "" = all manipulation methods; e.g. "Deepfakes,Face2Face"
MAX_PAIRS       = 0      # 0 = all clips; 300 for a short trial run (~15 min)
FRAMES_PER_CLIP = 2      # 1-2 recommended: more frames per clip is pseudo-replication
CROP_SIZE       = 384    # 0 = full frames (30x disk, 5x slower; only for the crop control in Session 6)
CROP_MARGIN     = 0.35
JPEG_Q          = 90
WORKERS         = 4
SHARD           = ""     # "0/2" and "1/2" in two notebooks to halve the wall time
VOCAB           = "face8"

# Module 1 gate
GATE_SAMPLE     = 40     # clips to probe
GATE_PROBES     = 3      # frames per probed clip
MIN_ALIGNMENT   = 0.90   # any change to this threshold must be reported with the results

# Quality gates for Module 2
MIN_IOU         = 0.70   # real-vs-fake parse agreement
MIN_COVERAGE    = 0.50   # fraction of changed pixels the vocabulary can name

# 8 h for parsing leaves about 2 h of head-room under the 10 h wall
TIME_BUDGET_MIN = 480
RESUME_FROM     = ""     # "/kaggle/input/<mount>/audit/parsed/index.json"

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
# ------------------------------------------------------- dependencies
KU.pip_install("mediapipe")
import importlib
from ccaudit import m2_parse as M2
importlib.reload(M2)
print("landmarker:", M2.locate_landmarker(""))

In [ ]:
# ------------------------------------------------------- locate the data
DATA = DATA or KU.find_dataset_root() or (KU.find_celebdf_root() or "")
if not DATA:
    raise SystemExit(
        "Could not find the dataset. Add Input -> your FF++ mirror "
        "(<your-ff-mirror>) or Celeb-DF mirror (<your-celebdf-mirror>), or set "
        "DATA to the folder that directly contains original/ and the "
        "manipulation folders.")
print("DATA =", DATA)
print("contains:", sorted(os.listdir(DATA))[:12])

In [ ]:
# ============================ MODULE 1: PAIRING GATE ============================
# Every downstream computation assumes that fake frame i and real frame i show
# the same instant.  This is verified on the actual video before any further
# cost is incurred.
PAIRS = f"{OUT}/pairs.json"
rc = KU.sh(
    f'{sys.executable} -m ccaudit.m1_verify --root "{DATA}" '
    f'--layout {LAYOUT} --methods "{METHODS}" --sample {GATE_SAMPLE} '
    f'--probes {GATE_PROBES} --min-alignment {MIN_ALIGNMENT} '
    f'--json "{OUT}/m1_report.json" --emit-pairs "{PAIRS}" --fail-hard',
    check=False, log=f"{OUT}/logs/m1.log")
if rc != 0:
    raise SystemExit(
        "PAIRING GATE FAILED. The PHASE 5 block above lists the diagnosis in "
        "order. Nothing should be parsed or audited until the gate passes: a "
        "failed gate means the counterfactual pairing does not hold.")

In [ ]:
# ============================ MODULE 2: PARSE ============================
# The parser uses the mediapipe Tasks API only.  There is deliberately no
# geometric fallback, because a fallback can silently produce boxes that do
# not sit on the face.
resume = f' --resume-from "{RESUME_FROM}"' if RESUME_FROM else ""
shard  = f' --shard {SHARD}' if SHARD else ""
KU.sh(
    f'{sys.executable} -m ccaudit.m2_parse --pairs "{PAIRS}" '
    f'--out "{OUT}/parsed" --data-root "{DATA}" '
    f'--frames-per-clip {FRAMES_PER_CLIP} --crop-size {CROP_SIZE} '
    f'--crop-margin {CROP_MARGIN} --jpeg-q {JPEG_Q} --min-iou {MIN_IOU} '
    f'--min-coverage {MIN_COVERAGE} --methods "{METHODS}" '
    f'--max-pairs {MAX_PAIRS} --workers {WORKERS} --vocab {VOCAB} '
    f'--time-budget-min {TIME_BUDGET_MIN}{shard}{resume} '
    f'--overlay "{OUT}/overlay.png"',
    check=False, log=f"{OUT}/logs/m2.log")

In [ ]:
# ------------------------------------- VISUAL CHECK OF THE PARSE
# If the coloured regions do not sit on the face, the parse is wrong and every
# downstream number is meaningless.  The log should contain
#   "mediapipe backend ready (tasks API)"
try:
    from IPython.display import Image, display
    if os.path.exists(f"{OUT}/overlay.png"):
        display(Image(filename=f"{OUT}/overlay.png"))
    else:
        print("no overlay written -- module 2 produced no samples")
except Exception as exc:                 # a display failure must not abort a long run
    print(f"(could not display the overlay: {exc}); the file is at "
          f"{OUT}/overlay.png -- open it from the Output panel)")

In [ ]:
# ------------------------------------------------------- summary
stats = C.load_json(f"{OUT}/parsed/parse_stats.json", {})
for k in ("n_records", "n_clips_with_output", "parser_iou_mean",
          "manip_coverage_mean", "n_dropped_total", "split_counts",
          "method_counts", "stopped_early"):
    print(f"  {k:24s} {stats.get(k)}")
print("  drops:", stats.get("drops"))
if stats.get("stopped_early"):
    print("\n!! Module 2 stopped on its time budget. Save this version, add "
          "its output as an input, set RESUME_FROM to its parsed/index.json "
          "and run again. Finished clips are skipped.")

In [ ]:
NEXT_STEP = """1. Save Version -> Save & Run All (Commit) if not already done.
2. Open the finished version -> Output tab -> New Dataset. Name it
   `cca-s1-parsed`. That dataset is the input to every later session.
3. Continue with Session 2 (02_cpu_controls.ipynb).

Before moving on, confirm:
  * Module 1 printed RESULT: PAIRING VERIFIED
  * the overlay picture shows regions sitting on the face
  * parser IoU mean is above ~0.7 and coverage above ~0.5
  * `samples written` is roughly 2 x the number of clips"""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)